# 08 - Monte Carlo robustness under renewable and demand uncertainty (workflow step 11a)

**Question:** are the nominal differences robust when PV, wind and demand deviate from their
forecasts, and does preserving flexibility (the uncertainty reserve) pay off in the tails?

$N_{MC}=100$ independent 24-h realizations (AR(1) forecast errors, seed 2026) -- never seen by the
EMS, whose reserve uses a separate sample set. **Common random numbers:** every strategy faces
exactly the same 100 realizations, so differences are paired (Wilcoxon signed-rank tests).

Outputs: `output/results/results_mc_trials.csv`, `output/results/results_mc_summary.csv`,
`output/results/results_mc_paired_tests.csv`, `output/experiment_figures/Fig_monte_carlo.pdf`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from dfm_acdc.config import RESULTS_DIR, FIG_DIR, MCParams
from dfm_acdc.simulate import Study
from dfm_acdc.mc import run_jobs
from dfm_acdc import plotstyle

plotstyle.apply_style()
PRE_PATH = os.path.join(RESULTS_DIR, "precompute_network_forecast.csv")
Study().load_or_precompute(PRE_PATH)
mc = MCParams()
if os.environ.get("DFM_SMOKE"):
    mc.n_trials = 2          # quick end-to-end smoke test only
STRATS = ["B0", "B1", "B2", "DFM"]

In [ ]:
t0 = time.perf_counter()
jobs = [{"strategy": s, "trial": t, "tag": "mc_main"} for t in range(mc.n_trials) for s in STRATS]
out = run_jobs(jobs, PRE_PATH, mc.seed, mc.n_trials)
trials = pd.DataFrame([o["summary"] for o in out]).sort_values(["strategy", "trial"]).reset_index(drop=True)
trials.to_csv(os.path.join(RESULTS_DIR, "results_mc_trials.csv"), index=False)
print(f"{len(trials)} runs in {time.perf_counter()-t0:.0f} s")

In [ ]:
METRICS = {"E_grid_corr_kWh": "E_grid (terminal-corrected) [kWh]", "cost_corr_EUR": "Energy cost (corrected) [EUR]",
           "E_curt_kWh": "E_curt [kWh]", "eta_RES": "eta_RES [-]", "E_loss_kWh": "E_loss [kWh]",
           "dSOH_rainflow": "Delta SOH [-]", "N_EFC": "N_EFC [-]", "E_dev_kWh": "Uncovered imbalance [kWh]",
           "n_viol_intervals": "Violating intervals [-]", "max_exceedance": "Max. normalized exceedance [-]",
           "soc_final": "Final SOC [-]", "t_solve_mean_ms": "Mean solve time [ms]"}
rows = []
for s in STRATS:
    d = trials[trials.strategy == s]
    for m in METRICS:
        x = d[m].values
        rows.append({"strategy": s, "metric": m, "mean": x.mean(), "std": x.std(ddof=1),
                     "P5": np.percentile(x, 5), "P50": np.percentile(x, 50), "P95": np.percentile(x, 95)})
    rows.append({"strategy": s, "metric": "P_viol_trials", "mean": float((d.n_viol_intervals > 0).mean()),
                 "std": np.nan, "P5": np.nan, "P50": np.nan, "P95": np.nan})
summ = pd.DataFrame(rows)
summ.to_csv(os.path.join(RESULTS_DIR, "results_mc_summary.csv"), index=False)
print(summ.pivot(index="metric", columns="strategy", values="mean")[STRATS].to_string())
print("\nP95:"); print(summ.pivot(index="metric", columns="strategy", values="P95")[STRATS].to_string())

## Paired comparison of DFM against each benchmark (same realizations)

In [ ]:
rows = []
dfm = trials[trials.strategy == "DFM"].set_index("trial")
for s in ("B0", "B1", "B2"):
    o = trials[trials.strategy == s].set_index("trial")
    for m in ("E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "E_dev_kWh", "n_viol_intervals"):
        if s == "B0" and m == "dSOH_rainflow":
            continue
        diff = (dfm[m] - o[m]).values
        try:
            p = wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0
        except ValueError:
            p = np.nan
        rows.append({"vs": s, "metric": m, "mean_diff": diff.mean(), "rel_diff_pct": 100 * diff.mean() / o[m].mean() if o[m].mean() != 0 else np.nan,
                     "share_DFM_better": float(np.mean(diff < 0)), "share_equal": float(np.mean(diff == 0)), "wilcoxon_p": p})
pt = pd.DataFrame(rows)
pt.to_csv(os.path.join(RESULTS_DIR, "results_mc_paired_tests.csv"), index=False)
print(pt.round(4).to_string(index=False))

## Figure 7 -- Monte Carlo distributions

In [ ]:
FIG_WIDTH, FIG_HEIGHT = plotstyle.IEEE_COL_WIDTH_IN * 1.55, plotstyle.IEEE_COL_WIDTH_IN * 1.55
fig, axs = plt.subplots(2, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))
panels = [("cost_corr_EUR", "Energy cost [EUR]", STRATS), ("dSOH_rainflow", r"$\Delta SOH$ [$10^{-5}$]", STRATS[1:]),
          ("E_curt_kWh", r"$E_{curt}$ [kWh]", STRATS), ("E_dev_kWh", "Uncovered imbalance [kWh]", STRATS)]
for ax, (m, lab, ss) in zip(axs.ravel(), panels):
    data = [trials[trials.strategy == s][m].values * (1e5 if m == "dSOH_rainflow" else 1) for s in ss]
    bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=True,
                    medianprops=dict(color=plotstyle.INK_PRIMARY, lw=1.0),
                    flierprops=dict(marker="o", ms=2.5, mfc=plotstyle.INK_MUTED, mec="none"))
    for patch, s in zip(bp["boxes"], ss):
        patch.set_facecolor(plotstyle.STRATEGY_COLORS[s]); patch.set_alpha(0.55); patch.set_edgecolor(plotstyle.INK_SECONDARY)
    ax.set_xticks(range(1, len(ss) + 1)); ax.set_xticklabels(ss)
    ax.set_ylabel(lab)
for a, lab in zip(axs.ravel(), "abcd"):
    a.text(-0.30, 1.04, f"({lab})", transform=a.transAxes, fontsize=8, va="top")
plotstyle.savefig(fig, os.path.join(FIG_DIR, "Fig_monte_carlo.pdf"))
plt.show()

## Network violations under uncertainty

In [ ]:
viol = trials.groupby("strategy").agg(trials_with_violation=("n_viol_intervals", lambda x: int((x > 0).sum())),
                                      mean_violating_intervals=("n_viol_intervals", "mean"),
                                      max_exceedance_P95=("max_exceedance", lambda x: np.percentile(x, 95)))
print(viol.loc[STRATS].to_string())